# DS2 Enrichment B: Multinomial Logistic Regression and GLMs

**Pairs with:** ISLP Ch. 4, sections 4.3.5 (Multinomial Logistic Regression) and 4.6 (Generalized Linear Models)

**Time:** ~40 min, optional, self-paced

**What you will do**
- Model an outcome with more than two categories: where the patient goes at discharge (home, rehab, long-term care, or died)
- Read relative risk ratios against "home" and turn one into a plain-English sentence
- Predict the probability of each destination for three example patients
- Model length of stay with a Gamma GLM (log link) and compare it with the linear regression from DS1

In [ ]:
# Setup: load the CADets dataset (release 1)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

DATA_FILE = "cadets_admissions_release1_core.csv"
DATA_URL = "https://raw.githubusercontent.com/Rotem-Gershon/cadets-notebooks/main/data/" + DATA_FILE

# Use a local copy if there is one; otherwise read the course copy from GitHub.
df = pd.read_csv(DATA_FILE if os.path.exists(DATA_FILE) else DATA_URL)
print(df.shape)

## Part 1: Multinomial logistic regression (ISLP 4.3.5)

### 1.1 The outcome: discharge destination

Logistic regression handles two outcomes. `discharge_destination` has four. Count them first. "home" is the most common and the most natural comparison, so it will be our **reference** category. Patients who died are kept: "died" is simply one of the four destinations.

In [ ]:
print(df["discharge_destination"].value_counts())
print((df["discharge_destination"].value_counts(normalize=True)).round(3))

### 1.2 Code the outcome as numbers, with home = 0

statsmodels' multinomial model (`mnlogit`) needs the outcome as numbers 0, 1, 2, 3. The category coded 0 becomes the reference, so we put "home" first.

In [ ]:
destinations = ["home", "rehab", "long_term_care", "died"]   # home first = reference
df["dest_code"] = pd.Categorical(df["discharge_destination"], categories=destinations).codes

# Check the coding: each destination should map to one number
print(df.groupby("discharge_destination")["dest_code"].first().sort_values())

### 1.3 Fit the model

We use four predictors: age, heart failure, active cancer, and oxygen saturation on admission. The model fits **three** logistic-type equations at once, each comparing one destination with home. The summary therefore has three blocks of coefficients.

In [ ]:
mn = smf.mnlogit("dest_code ~ age + heart_failure + active_cancer + spo2", data=df).fit(disp=0)

coefs = mn.params.copy()
coefs.columns = destinations[1:]   # name the columns: each is "vs home"
print(coefs.round(3))

### 1.4 Relative risk ratios

Exponentiating a coefficient gives a **relative risk ratio** (RRR): how a predictor changes the odds of going to that destination *rather than home*. It is read like an odds ratio. Values above 1 push patients away from home toward that destination. We also show p-values.

In [ ]:
rrr = np.exp(mn.params)
rrr.columns = destinations[1:]
pvals = mn.pvalues.copy()
pvals.columns = destinations[1:]

print("Relative risk ratios (vs home):")
print(rrr.drop(index="Intercept").round(2))
print("\nP-values:")
print(pvals.drop(index="Intercept").round(3))

**Plain-English reading.** Look at active cancer in the `long_term_care` column (about 1.9). In words: *"Among patients of the same age, heart-failure status and oxygen saturation, those with active cancer had roughly twice the odds of being discharged to long-term care rather than home."*

Note what this does **not** say: it is not "twice the risk of long-term care", and it says nothing about cause. Each RRR always compares one destination with the reference.

**Try it.** Write a one-sentence interpretation of the age RRR for `rehab`. Hint: the RRR is per year of age; `rrr.loc['age', 'rehab'] ** 10` gives the RRR per 10 years.

In [ ]:
# Your code here

### 1.5 Predicted probabilities for three example patients

The model gives each patient a probability for every destination, and the four probabilities always add up to 1. We invent three patients and compare them.

In [ ]:
patients = pd.DataFrame({
    "age":           [55, 82, 82],
    "heart_failure": [0,  1,  1],
    "active_cancer": [0,  0,  1],
    "spo2":          [97, 93, 88],
}, index=["A: 55y, well", "B: 82y, HF", "C: 82y, HF, cancer, low SpO2"])

pred = pd.DataFrame(np.asarray(mn.predict(patients)), index=patients.index, columns=destinations)
pred["total"] = pred.sum(axis=1)
print(pred.round(3))

Patient A goes home almost surely. As age, heart failure, cancer and low oxygen saturation add up, probability moves away from home, mostly toward death and, to a lesser extent, long-term care. For a discharge-planning tool, this table is the product: one probability per destination, ready to show to a case manager.

## Part 2: Generalized linear models for length of stay (ISLP 4.6)

### 2.1 The shape of length of stay

In DS1 we modeled `los_days` with ordinary linear regression. LOS is always positive and right-skewed: most stays are short, a few are very long. Look at the histogram.

In [ ]:
plt.hist(df["los_days"], bins=40)
plt.xlabel("Length of stay (days)"); plt.ylabel("Number of patients")
plt.show()
print(df["los_days"].describe().round(2))

### 2.2 Linear regression (as in DS1)

A linear model says each predictor **adds** a fixed number of days. We use age, heart failure, CKD and active cancer.

In [ ]:
los_formula = "los_days ~ age + heart_failure + ckd + active_cancer"

lin = smf.ols(los_formula, data=df).fit()
print(lin.params.round(3))

### 2.3 A Gamma GLM with a log link

A GLM lets us choose two things: the **family** (the shape of the outcome's spread) and the **link** (how predictors connect to the mean). For a positive, skewed, continuous outcome like LOS, a common choice is the Gamma family with a log link. With a log link, each predictor **multiplies** the mean stay instead of adding days, and the predicted mean can never go below zero.

In [ ]:
import statsmodels.api as sm

gamma = smf.glm(los_formula, data=df,
                family=sm.families.Gamma(link=sm.families.links.Log())).fit()
print(gamma.summary().tables[1])

### 2.4 Reading the Gamma coefficients: exp(coef)

Exponentiate each coefficient to get a multiplicative effect on mean LOS. A value of 1.15 means "15% longer stays, other predictors held fixed". We print it as a percent change.

In [ ]:
effects = pd.DataFrame({
    "exp_coef": np.exp(gamma.params),
    "pct_change_in_mean_LOS": (np.exp(gamma.params) - 1) * 100,
}).drop(index="Intercept")
print(effects.round(2))

hf = effects.loc["heart_failure", "pct_change_in_mean_LOS"]
print(f"\nHeart failure: stays are about {hf:.0f}% longer, on average.")

Compare with the linear model, which said heart failure adds a fixed number of days to everyone. The Gamma model says it adds a fixed **percentage**: more extra days for patients who already have long expected stays. Clinically, that is often the more natural assumption.

### 2.5 What about Poisson regression?

ISLP 4.6 uses the Bikeshare data, where the outcome is the **number of bikes rented** per hour. That is a count (0, 1, 2, ...), so ISLP uses a **Poisson** GLM with a log link. Poisson is designed for counts, for example the number of ED visits in a year or the number of falls on a ward. LOS in our data is measured in fractions of a day (for example 4.8), so it is a continuous positive measurement, and Gamma is the better fit. Both use the log link, so `exp(coef)` is read the same way: a multiplicative effect on the mean.

### 2.6 Predicted mean LOS by age, both models

For a patient without heart failure, CKD or cancer, plot the predicted mean LOS across ages. The straight line is the linear model; the curve is the Gamma GLM.

In [ ]:
ages = pd.DataFrame({"age": np.arange(20, 101),
                     "heart_failure": 0, "ckd": 0, "active_cancer": 0})

plt.plot(ages["age"], lin.predict(ages), label="Linear regression (adds days)")
plt.plot(ages["age"], gamma.predict(ages), label="Gamma GLM, log link (multiplies)")
plt.xlabel("Age (years)"); plt.ylabel("Predicted mean LOS (days)")
plt.legend(); plt.show()

In the middle of the age range the two models agree closely. They differ at the edges, where the Gamma curve bends. When models agree, choose the one whose assumptions and outputs (days added vs. percent longer) are easier to explain to the people who will use the product.

**Try it.** Add `heart_failure` = 1 to the `ages` table (set the column to 1) and redraw. In which model does the gap between heart failure and no heart failure grow with age?

In [ ]:
# Your code here

## Key points

- Multinomial logistic regression handles outcomes with more than two categories by comparing each category with a reference (here, home).
- `exp(coef)` gives a relative risk ratio: the change in odds of that destination *rather than home*. Always name the reference in your sentence.
- Predicted probabilities across all destinations add up to 1 for every patient, which makes them easy to display in a tool.
- A GLM chooses a family and a link. For positive, skewed LOS, a Gamma GLM with log link gives multiplicative effects ("heart failure: about 15% longer stays") and never predicts negative stays.
- Poisson GLMs are for counts (ISLP's Bikeshare example); Gamma suits continuous positive measurements like LOS.

**Further reading:** ISLP Ch. 4, section 4.3.5 (Multinomial Logistic Regression) and section 4.6 (Generalized Linear Models), including Lab 4.7.7 (Poisson Regression on the Bikeshare data).